In [46]:
import dotenv
import os
from tickflow import TickFlow

# 加载环境变量
dotenv.load_dotenv()

# 使用免费服务（无需 API key）
tf = TickFlow(api_key=os.getenv("TICKFLOW_APIKEY"))

# 查询日K线数据
df = tf.klines.get("600000.SH", period="1m", count=100, as_dataframe=True)
print(df.tail())

       symbol  name      timestamp  trade_date           trade_time  open  \
95  600000.SH  浦发银行  1777013760000  2026-04-24  2026-04-24 14:56:00  9.44   
96  600000.SH  浦发银行  1777013820000  2026-04-24  2026-04-24 14:57:00  9.44   
97  600000.SH  浦发银行  1777013880000  2026-04-24  2026-04-24 14:58:00  9.45   
98  600000.SH  浦发银行  1777013940000  2026-04-24  2026-04-24 14:59:00  9.45   
99  600000.SH  浦发银行  1777014000000  2026-04-24  2026-04-24 15:00:00  9.45   

    high   low  close  volume     amount  
95  9.45  9.44   9.45    5245  4954446.0  
96  9.45  9.44   9.44    7163  6763341.0  
97  9.45  9.45   9.45      53    50063.0  
98  9.45  9.45   9.45       0        0.0  
99  9.45  9.45   9.45    9332  8818740.0  


In [2]:
import datetime

symbol = "600519.SH"

factors_df = tf.klines.ex_factors([symbol], as_dataframe=True)
latest_factor = factors_df.iloc[-1]
print(f"最近除权日: {latest_factor['trade_date']}, 因子: {latest_factor['ex_factor']:.6f}")


最近除权日: 2025-12-19, 因子: 1.017003


In [4]:
factors_df.head()

,symbol,timestamp,trade_date,ex_factor
0,600519.SH,1027526400000,2002-07-25,1.118279
1,600519.SH,1058112000000,2003-07-14,1.108714
2,600519.SH,1088611200000,2004-07-01,1.311507
3,600519.SH,1123171200000,2005-08-05,1.210951
4,600519.SH,1147968000000,2006-05-19,2.006365


In [6]:
from pathlib import Path
import sys
from datetime import datetime, timedelta

import pandas as pd

# 兼容不同 notebook 工作目录，确保可导入本地脚本模块
for candidate in [Path.cwd(), Path.cwd() / "python_scripts"]:
    if (candidate / "insert_stock_daily.py").exists():
        candidate_str = str(candidate)
        if candidate_str not in sys.path:
            sys.path.insert(0, candidate_str)
        break

from insert_stock_daily import (
    DDB_DB_PATH,
    DDB_TABLE_NAME,
    connect_dolphindb_session,
    ensure_dolphindb_table,
    get_all_stock_symbols,
    get_ex_factors_batch,
    get_history_klines_batch,
    write_to_dolphindb,
)

print("导入完成，准备开始分步真实验证")

导入完成，准备开始分步真实验证


In [7]:
symbols = get_all_stock_symbols()
print(f"股票总数: {len(symbols)}")
print("前10只:", symbols[:10])

股票总数: 5513
前10只: ['600206.SH', '603693.SH', '605122.SH', '688244.SH', '603712.SH', '603059.SH', '603797.SH', '605116.SH', '600663.SH', '603200.SH']


In [8]:
sample_symbols = symbols[:3]
start_ts = int((datetime.now() - timedelta(days=365)).timestamp() * 1000)
end_ts = int(datetime.now().timestamp() * 1000)

kline_dict = get_history_klines_batch(sample_symbols, start_ts, end_ts)
print("测试标的:", sample_symbols)
for sym in sample_symbols:
    df_sym = kline_dict.get(sym, pd.DataFrame())
    print(f"\n{sym} 行数: {len(df_sym)}")
    if not df_sym.empty:
        print(df_sym.head(3).to_string(index=False))

测试标的: ['600206.SH', '603693.SH', '605122.SH']

600206.SH 行数: 242
     code name trade_date  open  high   low  close  volume       amount  adjust_factor
600206.SH      2025-04-28 18.28 18.29 17.73  17.74  431679  773128785.0       1.015283
600206.SH      2025-04-29 17.74 18.09 17.59  17.98  395808  708962754.0       1.015283
600206.SH      2025-04-30 18.25 18.37 17.90  18.05  594352 1075375640.0       1.015283

603693.SH 行数: 242
     code name trade_date  open  high   low  close  volume      amount  adjust_factor
603693.SH      2025-04-28 13.50 13.51 12.69  12.71  288291 373144565.0       1.012765
603693.SH      2025-04-29 12.47 12.52 11.81  12.00  264555 317112974.0       1.012765
603693.SH      2025-04-30 11.84 12.02 11.84  11.88  141078 168148131.0       1.012765

605122.SH 行数: 242
     code name trade_date  open  high   low  close  volume     amount  adjust_factor
605122.SH      2025-04-28 10.45 10.50 10.12  10.21   12493 12807521.0       1.001768
605122.SH      2025-04-29 10.36 10.

In [21]:
factor_map = get_ex_factors_batch(sample_symbols)
for sym in sample_symbols:
    fdf = factor_map.get(sym, pd.DataFrame())
    print(f"\n{sym} 因子行数: {len(fdf)}")
    if not fdf.empty:
        print(fdf.tail(3).to_string(index=False))


600206.SH 因子行数: 13
trade_date  ex_factor
2023-06-27   1.007874
2024-06-06   1.015283
2025-06-27   1.005726

603693.SH 因子行数: 8
trade_date  ex_factor
2023-07-07   1.011013
2024-06-27   1.012765
2025-06-20   1.011547

605122.SH 因子行数: 3
trade_date  ex_factor
2021-06-02   1.419874
2022-05-27   1.004742
2024-05-31   1.001768


In [24]:
s = connect_dolphindb_session()
ensure_dolphindb_table(s)

try:
    write_symbol = None
    write_df = pd.DataFrame()
    for sym in sample_symbols:
        candidate = kline_dict.get(sym, pd.DataFrame())
        if not candidate.empty:
            write_symbol = sym
            write_df = candidate.tail(5).copy()
            break

    if write_symbol is None:
        raise RuntimeError("样本股票没有可写入的数据")

    inserted_rows = write_to_dolphindb(write_df, s)
    print(f"写入标的: {write_symbol}, 写入行数: {inserted_rows}")

    s.upload({"dbPath": DDB_DB_PATH, "tableName": DDB_TABLE_NAME, "sym": write_symbol})
    verify_df = s.run(
        """
        select top 5 *
        from loadTable(dbPath, tableName)
        where code = sym
        order by trade_date desc
        """
    )
    print("回查最近5行:")
    print(verify_df)
finally:
    s.close()

DolphinDB 表 dfs://ohlcv_daily.stock_kline_daily 已准备就绪
写入标的: 600206.SH, 写入行数: 5
回查最近5行:
        code name trade_date   open   high    low  close  volume  \
0  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   
1  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   
2  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   
3  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   
4  600206.SH      2026-04-23  26.82  27.54  25.82  25.95  927279   

         amount  adjust_factor  
0  2.634101e+09       1.000000  
1  2.634101e+09       1.005726  
2  2.634101e+09       1.005726  
3  2.634101e+09       1.005726  
4  2.457931e+09       1.000000  


In [25]:
# 写库重试验证：连接断开时自动重连
max_retries = 3
inserted_rows = None
write_symbol = None
write_df = pd.DataFrame()

for sym in sample_symbols:
    candidate = kline_dict.get(sym, pd.DataFrame())
    if not candidate.empty:
        write_symbol = sym
        write_df = candidate.tail(5).copy()
        break

if write_symbol is None:
    raise RuntimeError("样本股票没有可写入的数据")

last_error = None
for attempt in range(1, max_retries + 1):
    s = connect_dolphindb_session()
    try:
        ensure_dolphindb_table(s)
        inserted_rows = write_to_dolphindb(write_df, s)
        s.upload({"dbPath": DDB_DB_PATH, "tableName": DDB_TABLE_NAME, "sym": write_symbol})
        verify_df = s.run(
            """
            select top 5 *
            from loadTable(dbPath, tableName)
            where code = sym
            order by trade_date desc
            """
        )
        print(f"第 {attempt} 次写入成功: {write_symbol}, 行数: {inserted_rows}")
        print("回查最近5行:")
        print(verify_df)
        break
    except Exception as e:
        last_error = e
        print(f"第 {attempt} 次写入失败: {e}")
    finally:
        try:
            s.close()
        except Exception:
            pass

if inserted_rows is None:
    raise RuntimeError(f"重试后仍写入失败: {last_error}")

DolphinDB 表 dfs://ohlcv_daily.stock_kline_daily 已准备就绪
第 1 次写入成功: 600206.SH, 行数: 5
回查最近5行:
        code name trade_date   open   high    low  close  volume  \
0  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   
1  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   
2  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   
3  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   
4  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   

         amount  adjust_factor  
0  2.634101e+09       1.000000  
1  2.634101e+09       1.005726  
2  2.634101e+09       1.005726  
3  2.634101e+09       1.005726  
4  2.634101e+09       1.005726  


In [26]:
import importlib
import insert_stock_daily as isd

importlib.reload(isd)

connect_dolphindb_session = isd.connect_dolphindb_session
ensure_dolphindb_table = isd.ensure_dolphindb_table
write_to_dolphindb = isd.write_to_dolphindb
DDB_DB_PATH = isd.DDB_DB_PATH
DDB_TABLE_NAME = isd.DDB_TABLE_NAME

print("已重载 insert_stock_daily，使用最新写入实现")

已重载 insert_stock_daily，使用最新写入实现


In [27]:
s = connect_dolphindb_session()
ensure_dolphindb_table(s)

try:
    write_symbol = None
    write_df = pd.DataFrame()
    for sym in sample_symbols:
        candidate = kline_dict.get(sym, pd.DataFrame())
        if not candidate.empty:
            write_symbol = sym
            write_df = candidate.tail(5).copy()
            break

    if write_symbol is None:
        raise RuntimeError("样本股票没有可写入的数据")

    inserted_rows = write_to_dolphindb(write_df, s)
    print(f"写入成功: {write_symbol}, 行数: {inserted_rows}")

    s.upload({"dbPath": DDB_DB_PATH, "tableName": DDB_TABLE_NAME, "sym": write_symbol})
    verify_df = s.run(
        """
        select top 5 *
        from loadTable(dbPath, tableName)
        where code = sym
        order by trade_date desc
        """
    )
    print("回查最近5行:")
    print(verify_df)
finally:
    s.close()

DolphinDB 表 dfs://ohlcv_daily.stock_kline_daily 已准备就绪
写入成功: 600206.SH, 行数: 5
回查最近5行:
        code name trade_date   open   high    low  close  volume  \
0  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   
1  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   
2  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   
3  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   
4  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   

         amount  adjust_factor  
0  2.634101e+09       1.000000  
1  2.634101e+09       1.005726  
2  2.634101e+09       1.005726  
3  2.634101e+09       1.005726  
4  2.634101e+09       1.005726  


In [28]:
# 回查验证（使用 upload + run 无参方式，避免 run(script, *args) 断连）
s = connect_dolphindb_session()
try:
    s.upload({"dbPath": DDB_DB_PATH, "tableName": DDB_TABLE_NAME, "sym": write_symbol})
    verify_df = s.run(
        """
        select top 5 *
        from loadTable(dbPath, tableName)
        where code = sym
        order by trade_date desc
        """
    )
    print(f"回查标的: {write_symbol}")
    print(verify_df)
finally:
    s.close()

回查标的: 600206.SH
        code name trade_date   open   high    low  close  volume  \
0  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   
1  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   
2  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   
3  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   
4  600206.SH      2026-04-24  25.73  27.17  25.26  26.62  997848   

         amount  adjust_factor  
0  2.634101e+09       1.000000  
1  2.634101e+09       1.005726  
2  2.634101e+09       1.005726  
3  2.634101e+09       1.005726  
4  2.634101e+09       1.005726  


In [29]:
import importlib
import insert_stock_daily as isd

importlib.reload(isd)
get_history_klines_batch = isd.get_history_klines_batch

kline_dict = get_history_klines_batch(sample_symbols, start_ts, end_ts)
for sym in sample_symbols:
    df_sym = kline_dict.get(sym, pd.DataFrame())
    print(f"\n{sym} adjust_factor 唯一值(前10):", sorted(df_sym["adjust_factor"].dropna().unique().tolist())[:10])
    if not df_sym.empty:
        print(df_sym[["trade_date", "adjust_factor"]].head(5).to_string(index=False))


600206.SH adjust_factor 唯一值(前10): [1.005726]
trade_date  adjust_factor
2026-03-30       1.005726
2026-03-31       1.005726
2026-04-01       1.005726
2026-04-02       1.005726
2026-04-03       1.005726

603693.SH adjust_factor 唯一值(前10): [1.011547]
trade_date  adjust_factor
2026-03-30       1.011547
2026-03-31       1.011547
2026-04-01       1.011547
2026-04-02       1.011547
2026-04-03       1.011547

605122.SH adjust_factor 唯一值(前10): [1.001768]
trade_date  adjust_factor
2026-03-30       1.001768
2026-03-31       1.001768
2026-04-01       1.001768
2026-04-02       1.001768
2026-04-03       1.001768


In [30]:
import random

sample_size = 20
symbols_pool = symbols if "symbols" in globals() and len(symbols) > 0 else get_all_stock_symbols()
check_symbols = random.sample(symbols_pool, min(sample_size, len(symbols_pool)))

check_start_ts = int((datetime.now() - timedelta(days=45)).timestamp() * 1000)
check_end_ts = int(datetime.now().timestamp() * 1000)

kline_check = get_history_klines_batch(check_symbols, check_start_ts, check_end_ts)
factor_check = get_ex_factors_batch(check_symbols)

rows = []
for sym in check_symbols:
    dfk = kline_check.get(sym, pd.DataFrame())
    fdf = factor_check.get(sym, pd.DataFrame())

    if dfk.empty:
        rows.append({"symbol": sym, "status": "no_kline", "expected": None, "observed": None, "match": False})
        continue

    last_trade_date = pd.to_datetime(dfk["trade_date"]).max().date()
    observed = float(dfk.sort_values("trade_date")["adjust_factor"].iloc[-1])

    expected = 1.0
    if fdf is not None and not fdf.empty:
        valid = fdf[pd.to_datetime(fdf["trade_date"]).dt.date <= last_trade_date]
        if not valid.empty:
            expected = float(valid.sort_values("trade_date")["ex_factor"].iloc[-1])

    match = abs(observed - expected) < 1e-9
    rows.append(
        {
            "symbol": sym,
            "status": "ok",
            "expected": expected,
            "observed": observed,
            "match": match,
        }
    )

check_df = pd.DataFrame(rows)
valid_df = check_df[check_df["status"] == "ok"]
pass_count = int(valid_df["match"].sum()) if not valid_df.empty else 0
print(f"抽样校验: {len(check_symbols)} 只, 有效样本: {len(valid_df)}, 一致: {pass_count}, 不一致: {len(valid_df) - pass_count}")

mismatch_df = valid_df[~valid_df["match"]]
if mismatch_df.empty:
    print("全部一致")
else:
    print("不一致样本:")
    print(mismatch_df.to_string(index=False))

check_df.head(20)

抽样校验: 20 只, 有效样本: 20, 一致: 20, 不一致: 0
全部一致


,symbol,status,expected,observed,match
0,688576.SH,ok,1.015982,1.015982,True
1,300225.SZ,ok,1.004672,1.004672,True
2,603153.SH,ok,1.013896,1.013896,True
3,301102.SZ,ok,1.001888,1.001888,True
4,600644.SH,ok,1.007153,1.007153,True
5,300150.SZ,ok,1.008174,1.008174,True
6,688575.SH,ok,1.014413,1.014413,True
7,001257.SZ,ok,1.000000,1.000000,True
8,300617.SZ,ok,1.008872,1.008872,True
9,300106.SZ,ok,1.000961,1.000961,True


In [35]:
df_5m = tf.klines.intraday("600000.SH", period="5m", as_dataframe=True)
df_5m["vwap"] = (df_5m["amount"] / df_5m["volume"]).round(2)
print(df_5m[["trade_time", "close", "vwap"]].tail())

             trade_time  close    vwap
43  2026-04-24 14:40:00   9.47  947.05
44  2026-04-24 14:45:00   9.47  946.39
45  2026-04-24 14:50:00   9.46  945.57
46  2026-04-24 14:55:00   9.45  944.43
47  2026-04-24 15:00:00   9.45  944.64


In [38]:
df_5m.head()

,symbol,name,timestamp,trade_date,trade_time,open,high,low,close,volume,amount,vwap
0,600000.SH,浦发银行,1776994500000,2026-04-24,2026-04-24 09:35:00,9.53,9.53,9.46,9.49,48270,45779788.0,948.41
1,600000.SH,浦发银行,1776994800000,2026-04-24,2026-04-24 09:40:00,9.48,9.49,9.47,9.48,29167,27645674.0,947.84
2,600000.SH,浦发银行,1776995100000,2026-04-24,2026-04-24 09:45:00,9.47,9.50,9.47,9.48,10460,9920413.0,948.41
3,600000.SH,浦发银行,1776995400000,2026-04-24,2026-04-24 09:50:00,9.48,9.49,9.47,9.48,14004,13277899.0,948.15
4,600000.SH,浦发银行,1776995700000,2026-04-24,2026-04-24 09:55:00,9.48,9.51,9.48,9.51,10036,9523062.0,948.89


In [44]:
# 导入库
from datetime import datetime

# 准备股票列表，需使用 '代码.市场' 的标准格式
symbols = ["000001.SZ", "600000.SH"]

# 发起批量请求
df_dict = tf.klines.intraday(
    symbol="600000.SH",
    period="60m",  # 或 "1m", "5m", "15m", "30m",
    count=1000,
    as_dataframe=True
)

In [47]:
df_dict.columns

Index(['symbol', 'name', 'timestamp', 'trade_date', 'trade_time', 'open',
       'high', 'low', 'close', 'volume', 'amount'],
      dtype='object')

In [55]:
import importlib
from datetime import datetime, timedelta

import pandas as pd
import insert_stock_intraday as isi

importlib.reload(isi)

print("已重载 insert_stock_intraday")
print("INTRADAY_PERIODS:", isi.INTRADAY_PERIODS)
print("INTRADAY_SYMBOL_LIMIT:", isi.INTRADAY_SYMBOL_LIMIT)

已重载 insert_stock_intraday
INTRADAY_PERIODS: 1m,5m,15m,30m,60m
INTRADAY_SYMBOL_LIMIT: 50


In [56]:
# Step 1: 原始拉取检查（batch -> period_map）
probe_symbols = isi.get_all_stock_symbols()[:5]
probe_periods = ["1m"]
probe_start_ts = int((datetime.now() - timedelta(days=3)).timestamp() * 1000)
probe_end_ts = int(datetime.now().timestamp() * 1000)

period_map = isi.get_intraday_klines_multi_period_batch(
    symbols=probe_symbols,
    start_ts=probe_start_ts,
    end_ts=probe_end_ts,
    periods=probe_periods,
)

print("probe_symbols:", probe_symbols)
for sym in probe_symbols:
    raw_df = pd.DataFrame(period_map.get("1m", {}).get(sym, pd.DataFrame()))
    print(f"{sym}: 原始 rows={len(raw_df)}, columns={list(raw_df.columns)[:8]}")
    if not raw_df.empty:
        print(raw_df.head(2).to_string(index=False))

probe_symbols: ['600206.SH', '603693.SH', '605122.SH', '688244.SH', '603712.SH']
600206.SH: 原始 rows=241, columns=['symbol', 'name', 'timestamp', 'trade_date', 'trade_time', 'open', 'high', 'low']
   symbol name     timestamp trade_date          trade_time  open  high   low  close  volume     amount
600206.SH 有研新材 1776994200000 2026-04-24 2026-04-24 09:30:00 25.73 25.73 25.73  25.73    6108 15715884.0
600206.SH 有研新材 1776994260000 2026-04-24 2026-04-24 09:31:00 25.73 26.11 25.61  26.10   13639 35239908.0
603693.SH: 原始 rows=241, columns=['symbol', 'name', 'timestamp', 'trade_date', 'trade_time', 'open', 'high', 'low']
   symbol name     timestamp trade_date          trade_time  open  high   low  close  volume     amount
603693.SH 江苏新能 1776994200000 2026-04-24 2026-04-24 09:30:00 15.91 15.91 15.91  15.91    3870  6157170.0
603693.SH 江苏新能 1776994260000 2026-04-24 2026-04-24 09:31:00 15.91 16.00 15.88  15.89   14772 23516167.0
605122.SH: 原始 rows=241, columns=['symbol', 'name', 'timestamp', '

In [57]:
# Step 2: 展平 + 因子合并检查
flat_df = isi._flatten_period_result(period_map.get("1m", {}))
factor_map = isi.get_ex_factors_batch(probe_symbols)
with_factor_df = isi._apply_ex_factors_to_intraday(flat_df, factor_map)

print("flatten rows:", len(flat_df))
print("with_factor rows:", len(with_factor_df))
print("adjust_factor unique sample:", sorted(with_factor_df.get("adjust_factor", pd.Series(dtype=float)).dropna().unique().tolist())[:5])
print(with_factor_df.head(3).to_string(index=False))

flatten rows: 1205
with_factor rows: 1205
adjust_factor unique sample: [1.0, 1.481997]
   symbol name     timestamp trade_date          trade_time  open  high   low  close  volume     amount  adjust_factor
600206.SH 有研新材 1776994200000 2026-04-24 2026-04-24 09:30:00 25.73 25.73 25.73  25.73    6108 15715884.0       1.481997
600206.SH 有研新材 1776994260000 2026-04-24 2026-04-24 09:31:00 25.73 26.11 25.61  26.10   13639 35239908.0       1.481997
600206.SH 有研新材 1776994320000 2026-04-24 2026-04-24 09:32:00 26.10 26.42 26.00  26.39   18019 47203548.0       1.481997


In [58]:
# Step 3: 标准化入库数据检查
norm_df = isi._normalize_intraday_for_dolphindb(with_factor_df)
print("normalize rows:", len(norm_df))
print("normalize columns:", norm_df.columns.tolist())
print(norm_df.head(3).to_string(index=False))

# Step 4: 实际写库 + 回查
s = isi.connect_dolphindb_session()
try:
    isi.ensure_intraday_table(s, "1m")
    inserted = isi.write_intraday_to_dolphindb(with_factor_df, "1m", s)
    print("tableInsert rows:", inserted)

    if not norm_df.empty:
        probe_code = norm_df.iloc[0]["code"]
        s.upload({"dbPath": isi.DDB_DB_PATH, "tableName": isi._intraday_table_name("1m"), "probeCode": probe_code})
        verify = s.run(
            """
            select top 5 *
            from loadTable(dbPath, tableName)
            where code = probeCode
            order by trade_time desc
            """
        )
        print("回查 rows:", len(verify))
        print(verify)
finally:
    s.close()

normalize rows: 1205
normalize columns: ['code', 'trade_date', 'trade_time', 'open', 'high', 'low', 'close', 'volume', 'amount', 'adjust_factor']
     code trade_date          trade_time  open  high   low  close  volume     amount  adjust_factor
600206.SH 2026-04-24 2026-04-24 01:30:00 25.73 25.73 25.73  25.73    6108 15715884.0       1.481997
600206.SH 2026-04-24 2026-04-24 01:31:00 25.73 26.11 25.61  26.10   13639 35239908.0       1.481997
600206.SH 2026-04-24 2026-04-24 01:32:00 26.10 26.42 26.00  26.39   18019 47203548.0       1.481997
tableInsert rows: 1205
回查 rows: 5
        code trade_date          trade_time   open   high    low  close  \
0  600206.SH 2026-04-24 2026-04-24 07:00:00  26.62  26.62  26.62  26.62   
1  600206.SH 2026-04-24 2026-04-24 06:59:00  26.61  26.61  26.61  26.61   
2  600206.SH 2026-04-24 2026-04-24 06:58:00  26.61  26.61  26.61  26.61   
3  600206.SH 2026-04-24 2026-04-24 06:57:00  26.60  26.63  26.59  26.61   
4  600206.SH 2026-04-24 2026-04-24 06:56:00  

In [53]:
import importlib
import insert_stock_daily as isd

importlib.reload(isd)
print("insert_stock_daily path:", isd.__file__)
print("has get_intraday_klines_multi_period_batch:", hasattr(isd, "get_intraday_klines_multi_period_batch"))
print("available names sample:", [n for n in dir(isd) if "intraday" in n.lower() or "kline" in n.lower()])

insert_stock_daily path: /home/caopozhi/projects/data_manager/python_scripts/insert_stock_daily.py
has get_intraday_klines_multi_period_batch: True
available names sample: ['INTRADAY_BATCH_INTERVAL_SEC', 'INTRADAY_BATCH_RPM', 'INTRADAY_MAX_COUNT_PER_REQ', 'KLINE_COLUMNS', 'KLINE_MAX_COUNT_PER_REQ', 'get_history_klines_batch', 'get_intraday_klines_multi_period_batch']
